# Chapter 3.9: Multi-Source Retrieval Fusion

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** multi-channel recall architecture: CF, content, trending, personalized channels
2. **Implement** score normalization methods: min-max, z-score, rank-based
3. **Design** quota allocation strategies: fixed vs dynamic allocation
4. **Describe** Meta's multi-source retrieval architecture
5. **Implement** de-duplication and diversity enforcement in retrieval fusion
6. **Build** a complete multi-channel fusion pipeline with dynamic quotas
7. **Evaluate** fusion quality and analyze channel contributions

## Prerequisites

- Retrieval methods from Chapters 3.1-3.7
- Pre-ranking concepts (Chapter 3.8)
- Basic statistics (normalization, ranking)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part3/chapter_3.9_retrieval_fusion.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part3/chapter_3.9_retrieval_fusion.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Set
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")

## 1. Multi-Channel Retrieval Architecture

Production recommendation systems use **multiple retrieval channels** (also called "recall sources"):

| Channel | Method | Captures | Example |
|---------|--------|----------|---------|
| **CF** | User-based / Item-based CF | Collaborative patterns | "Users like you also liked..." |
| **Embedding** | Two-tower / ANN | Semantic similarity | Deep learning retrieval |
| **Content** | TF-IDF / Semantic match | Content relevance | "Similar products" |
| **Trending** | Popularity-based | Current trends | "Hot right now" |
| **Personal Hot** | User-specific trending | Personal trends | "Trending in your feed" |
| **Sequential** | SASRec / Session-based | Recent intent | "Because you just viewed..." |
| **Graph** | GraphSAGE / PinSage | Network effects | "Friends are engaging with..." |

The challenge: How to **merge** candidates from different channels into a single ranked list.

> **💡 Concept:** No single retrieval channel is perfect. CF misses cold-start items, content-based misses collaborative signals, trending has no personalization. Multi-source fusion combines their complementary strengths.

In [ ]:
# Simulate multi-channel retrieval
NUM_ITEMS = 10000
NUM_CATEGORIES = 20
item_categories = np.random.randint(0, NUM_CATEGORIES, NUM_ITEMS)
item_popularity = np.random.power(0.3, NUM_ITEMS)  # Power-law popularity
item_popularity = item_popularity / item_popularity.max()

class RetrievalChannel:
    """Base class for a retrieval channel."""
    def __init__(self, name: str, num_items: int):
        self.name = name
        self.num_items = num_items
    
    def retrieve(self, user_id: int, k: int) -> List[Tuple[int, float]]:
        """Retrieve top-k items with scores."""
        raise NotImplementedError

class CFChannel(RetrievalChannel):
    """Collaborative filtering channel."""
    def __init__(self, num_items, user_item_matrix):
        super().__init__('CF', num_items)
        self.matrix = user_item_matrix
    
    def retrieve(self, user_id, k):
        scores = self.matrix[user_id] + np.random.randn(self.num_items) * 0.1
        top_k = np.argsort(-scores)[:k]
        return [(int(idx), float(scores[idx])) for idx in top_k]

class ContentChannel(RetrievalChannel):
    """Content-based similarity channel."""
    def __init__(self, num_items, item_features):
        super().__init__('Content', num_items)
        self.features = item_features
    
    def retrieve(self, user_id, k, query_features=None):
        if query_features is None:
            query_features = np.random.randn(self.features.shape[1])
        scores = self.features @ query_features
        scores += np.random.randn(self.num_items) * 0.1
        top_k = np.argsort(-scores)[:k]
        return [(int(idx), float(scores[idx])) for idx in top_k]

class TrendingChannel(RetrievalChannel):
    """Trending/popularity channel."""
    def __init__(self, num_items, popularity):
        super().__init__('Trending', num_items)
        self.popularity = popularity
    
    def retrieve(self, user_id, k):
        # Add noise for variability
        scores = self.popularity + np.random.randn(self.num_items) * 0.05
        top_k = np.argsort(-scores)[:k]
        return [(int(idx), float(scores[idx])) for idx in top_k]

class EmbeddingChannel(RetrievalChannel):
    """Deep embedding retrieval channel."""
    def __init__(self, num_items, item_embeddings, user_embeddings):
        super().__init__('Embedding', num_items)
        self.item_emb = item_embeddings / np.linalg.norm(item_embeddings, axis=1, keepdims=True)
        self.user_emb = user_embeddings / np.linalg.norm(user_embeddings, axis=1, keepdims=True)
    
    def retrieve(self, user_id, k):
        scores = self.item_emb @ self.user_emb[user_id]
        top_k = np.argsort(-scores)[:k]
        return [(int(idx), float(scores[idx])) for idx in top_k]

# Create channels with synthetic data
NUM_USERS = 200
DIM = 32

# CF matrix (sparse)
user_item_cf = np.random.randn(NUM_USERS, NUM_ITEMS).astype(np.float32) * 0.1
for u in range(NUM_USERS):
    prefs = np.random.choice(NUM_CATEGORIES, 3, replace=False)
    for cat in prefs:
        mask = item_categories == cat
        user_item_cf[u, mask] += np.random.rand() * 2

item_features = np.random.randn(NUM_ITEMS, DIM).astype(np.float32)
item_emb = np.random.randn(NUM_ITEMS, DIM).astype(np.float32)
user_emb = np.random.randn(NUM_USERS, DIM).astype(np.float32)

channels = {
    'CF': CFChannel(NUM_ITEMS, user_item_cf),
    'Content': ContentChannel(NUM_ITEMS, item_features),
    'Trending': TrendingChannel(NUM_ITEMS, item_popularity),
    'Embedding': EmbeddingChannel(NUM_ITEMS, item_emb, user_emb),
}

# Test
for name, channel in channels.items():
    results = channel.retrieve(0, 5)
    print(f"{name:10s}: {[(item, f'{score:.3f}') for item, score in results]}")

## 2. Score Normalization

Different channels produce scores on different scales. Before merging, we must normalize:

### Min-Max Normalization
$$s_{\text{norm}} = \frac{s - s_{\min}}{s_{\max} - s_{\min}}$$

### Z-Score Normalization
$$s_{\text{norm}} = \frac{s - \mu}{\sigma}$$

### Rank-Based Normalization
$$s_{\text{norm}} = 1 - \frac{\text{rank}(s)}{N}$$

| Method | Pros | Cons |
|--------|------|------|
| Min-Max | Bounded [0,1] | Sensitive to outliers |
| Z-Score | Robust to distribution | Not bounded |
| Rank-Based | Distribution-free | Loses magnitude info |

> **⚠️ Common Pitfall:** Normalization should be computed per-channel, per-request (not globally). The score distribution changes across different queries/users.

In [ ]:
def min_max_normalize(scores: np.ndarray) -> np.ndarray:
    """Min-max normalization to [0, 1]."""
    s_min, s_max = scores.min(), scores.max()
    if s_max - s_min < 1e-8:
        return np.ones_like(scores) * 0.5
    return (scores - s_min) / (s_max - s_min)

def z_score_normalize(scores: np.ndarray) -> np.ndarray:
    """Z-score normalization."""
    mu, sigma = scores.mean(), scores.std()
    if sigma < 1e-8:
        return np.zeros_like(scores)
    return (scores - mu) / sigma

def rank_normalize(scores: np.ndarray) -> np.ndarray:
    """Rank-based normalization to [0, 1]."""
    n = len(scores)
    ranks = np.argsort(np.argsort(-scores))  # Rank 0 = highest
    return 1.0 - ranks / n

# Demonstrate
user_id = 0
k_per_channel = 200

all_results = {}
for name, channel in channels.items():
    results = channel.retrieve(user_id, k_per_channel)
    items, scores = zip(*results)
    all_results[name] = (np.array(items), np.array(scores))

# Visualize score distributions before and after normalization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
norm_methods = {'Raw': lambda x: x, 'Min-Max': min_max_normalize, 
                'Z-Score': z_score_normalize, 'Rank': rank_normalize}

for ax, (norm_name, norm_fn) in zip(axes.flat, norm_methods.items()):
    for ch_name, (items, scores) in all_results.items():
        normalized = norm_fn(scores)
        ax.hist(normalized, bins=30, alpha=0.4, label=ch_name)
    ax.set_title(f'{norm_name} Scores', fontsize=12)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Score Distributions by Normalization Method', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Quota Allocation Strategies

Given a total retrieval budget of $N$ items, how many items should each channel contribute?

### Fixed Quota
Pre-defined allocation: e.g., CF=40%, Embedding=30%, Content=20%, Trending=10%.

### Dynamic Quota
Adjust quotas based on:
- **User type**: New users get more trending, returning users get more CF
- **Context**: Time of day, device, etc.
- **Channel confidence**: Channels with higher score variance get more budget
- **Historical performance**: Track which channels contribute to downstream engagement

> **🔑 Pro Tip:** In Meta's system, each channel has a minimum quota (for diversity) and a dynamic portion (for optimization). This ensures no channel is starved while allowing the system to adapt.

In [ ]:
class QuotaAllocator:
    """Allocate retrieval quotas to channels."""
    
    def __init__(self, channel_names: List[str]):
        self.channel_names = channel_names
    
    def fixed_quota(self, total_budget: int, 
                    ratios: Dict[str, float] = None) -> Dict[str, int]:
        """Fixed ratio allocation."""
        if ratios is None:
            ratios = {name: 1.0 / len(self.channel_names) for name in self.channel_names}
        
        allocations = {}
        remaining = total_budget
        for i, name in enumerate(self.channel_names):
            if i == len(self.channel_names) - 1:
                allocations[name] = remaining
            else:
                alloc = int(total_budget * ratios[name])
                allocations[name] = alloc
                remaining -= alloc
        return allocations
    
    def dynamic_quota(self, total_budget: int, 
                      channel_scores: Dict[str, np.ndarray],
                      min_ratio: float = 0.1) -> Dict[str, int]:
        """Dynamic allocation based on channel confidence (score variance)."""
        # Higher variance = more confident/discriminative = more budget
        variances = {}
        for name, scores in channel_scores.items():
            variances[name] = np.var(scores) + 1e-8
        
        total_var = sum(variances.values())
        
        # Minimum allocation
        min_alloc = int(total_budget * min_ratio)
        dynamic_budget = total_budget - min_alloc * len(self.channel_names)
        
        allocations = {}
        remaining = dynamic_budget
        for i, name in enumerate(self.channel_names):
            ratio = variances[name] / total_var
            if i == len(self.channel_names) - 1:
                dynamic = remaining
            else:
                dynamic = int(dynamic_budget * ratio)
                remaining -= dynamic
            allocations[name] = min_alloc + dynamic
        
        return allocations

# Demo
allocator = QuotaAllocator(list(channels.keys()))

fixed = allocator.fixed_quota(500, {'CF': 0.35, 'Content': 0.25, 'Trending': 0.15, 'Embedding': 0.25})
print("Fixed quota:", fixed)

channel_scores = {name: scores for name, (items, scores) in all_results.items()}
dynamic = allocator.dynamic_quota(500, channel_scores, min_ratio=0.1)
print("Dynamic quota:", dynamic)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (title, allocs) in zip(axes, [('Fixed Quota', fixed), ('Dynamic Quota', dynamic)]):
    names_q = list(allocs.keys())
    values_q = list(allocs.values())
    colors_q = plt.cm.Set2(np.linspace(0, 1, len(names_q)))
    ax.pie(values_q, labels=[f"{n}\n({v})" for n, v in zip(names_q, values_q)],
           autopct='%1.0f%%', colors=colors_q, startangle=90)
    ax.set_title(title, fontsize=13)

plt.suptitle('Quota Allocation (Total Budget = 500)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Fusion and De-duplication

When merging candidates from multiple channels:

1. **De-duplication**: Items may appear in multiple channels
2. **Score aggregation**: For duplicates, combine scores (max, avg, weighted)
3. **Diversity enforcement**: Prevent a single channel from dominating

### Score Fusion Strategies

For item $i$ appearing in channels $\mathcal{C}_i$:

- **Max fusion**: $s_i = \max_{c \in \mathcal{C}_i} s_{i,c}$
- **Average fusion**: $s_i = \frac{1}{|\mathcal{C}_i|} \sum_{c} s_{i,c}$
- **Weighted fusion**: $s_i = \sum_{c} w_c \cdot s_{i,c}$
- **Reciprocal rank fusion**: $s_i = \sum_{c} \frac{1}{k + \text{rank}_{c}(i)}$

In [ ]:
class RetrievalFusion:
    """Fuse results from multiple retrieval channels."""
    
    def __init__(self, normalization: str = 'rank', 
                 fusion_method: str = 'weighted'):
        self.normalization = normalization
        self.fusion_method = fusion_method
    
    def _normalize(self, scores: np.ndarray) -> np.ndarray:
        if self.normalization == 'min_max':
            return min_max_normalize(scores)
        elif self.normalization == 'z_score':
            return z_score_normalize(scores)
        elif self.normalization == 'rank':
            return rank_normalize(scores)
        return scores
    
    def fuse(self, channel_results: Dict[str, List[Tuple[int, float]]],
             channel_weights: Dict[str, float] = None,
             top_k: int = 100) -> List[Tuple[int, float, str]]:
        """
        Fuse results from multiple channels.
        
        Args:
            channel_results: {channel_name: [(item_id, score), ...]}
            channel_weights: {channel_name: weight} for weighted fusion
            top_k: number of items to return
        
        Returns:
            List of (item_id, fused_score, contributing_channels)
        """
        if channel_weights is None:
            channel_weights = {name: 1.0 for name in channel_results}
        
        # Normalize scores per channel
        normalized_results = {}
        for name, results in channel_results.items():
            if not results:
                continue
            items, scores = zip(*results)
            norm_scores = self._normalize(np.array(scores))
            normalized_results[name] = dict(zip(items, norm_scores))
        
        # Aggregate scores per item
        item_scores = defaultdict(list)  # item_id -> [(channel, score)]
        for name, item_score_dict in normalized_results.items():
            for item_id, score in item_score_dict.items():
                item_scores[item_id].append((name, score))
        
        # Fuse
        fused = []
        for item_id, channel_score_list in item_scores.items():
            if self.fusion_method == 'max':
                best = max(channel_score_list, key=lambda x: x[1])
                fused_score = best[1]
            elif self.fusion_method == 'average':
                fused_score = np.mean([s for _, s in channel_score_list])
            elif self.fusion_method == 'weighted':
                fused_score = sum(
                    channel_weights.get(ch, 1.0) * s 
                    for ch, s in channel_score_list
                ) / sum(
                    channel_weights.get(ch, 1.0) 
                    for ch, _ in channel_score_list
                )
            elif self.fusion_method == 'rrf':  # Reciprocal Rank Fusion
                fused_score = sum(
                    1.0 / (60 + (1 - s) * 100)  # Convert score to rank-like
                    for _, s in channel_score_list
                )
            else:
                raise ValueError(f"Unknown fusion: {self.fusion_method}")
            
            channels_str = '+'.join([ch for ch, _ in channel_score_list])
            fused.append((item_id, fused_score, channels_str))
        
        # Sort by fused score
        fused.sort(key=lambda x: -x[1])
        return fused[:top_k]


# Retrieve and fuse
user_id = 0
channel_results = {}
quotas = allocator.fixed_quota(300, {'CF': 0.35, 'Content': 0.25, 'Trending': 0.15, 'Embedding': 0.25})

for name, channel in channels.items():
    k = quotas[name]
    channel_results[name] = channel.retrieve(user_id, k)

# Try different fusion methods
fusion_methods = ['max', 'average', 'weighted', 'rrf']
weights = {'CF': 1.5, 'Content': 1.0, 'Trending': 0.5, 'Embedding': 1.2}

for method in fusion_methods:
    fusioner = RetrievalFusion(normalization='rank', fusion_method=method)
    results = fusioner.fuse(channel_results, weights, top_k=10)
    print(f"\n{method.upper()} fusion - Top 5:")
    for item_id, score, ch in results[:5]:
        print(f"  Item {item_id:5d} | Score: {score:.4f} | Channels: {ch}")

## 5. Diversity in Retrieval Fusion

After fusion, the result list may lack diversity. We enforce diversity using:

1. **Category diversity**: Limit items per category
2. **Channel diversity**: Ensure each channel contributes to the final list
3. **MMR (Maximal Marginal Relevance)**: Balance relevance with novelty

$$\text{MMR}(i) = \lambda \cdot \text{score}(i) - (1-\lambda) \cdot \max_{j \in S} \text{sim}(i, j)$$

In [ ]:
def diversified_selection(fused_results: List[Tuple[int, float, str]],
                          item_categories: np.ndarray,
                          max_per_category: int = 5,
                          max_per_channel: int = None,
                          target_size: int = 50) -> List[Tuple[int, float, str]]:
    """Select items with diversity constraints."""
    selected = []
    category_counts = defaultdict(int)
    channel_counts = defaultdict(int)
    
    for item_id, score, channels_str in fused_results:
        if len(selected) >= target_size:
            break
        
        cat = item_categories[item_id]
        
        # Category diversity check
        if category_counts[cat] >= max_per_category:
            continue
        
        # Channel diversity check (optional)
        if max_per_channel is not None:
            primary_channel = channels_str.split('+')[0]
            if channel_counts[primary_channel] >= max_per_channel:
                continue
            channel_counts[primary_channel] += 1
        
        category_counts[cat] += 1
        selected.append((item_id, score, channels_str))
    
    return selected


# Full fusion pipeline
fusioner = RetrievalFusion(normalization='rank', fusion_method='weighted')
fused = fusioner.fuse(channel_results, weights, top_k=500)

# Without diversity
no_div = fused[:50]
no_div_cats = [item_categories[item_id] for item_id, _, _ in no_div]

# With diversity
with_div = diversified_selection(fused, item_categories, max_per_category=5, target_size=50)
with_div_cats = [item_categories[item_id] for item_id, _, _ in with_div]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(no_div_cats, bins=range(NUM_CATEGORIES+1), color='coral', alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Category', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title(f'Without Diversity\n({len(set(no_div_cats))} unique categories)', fontsize=12)
axes[0].axhline(y=5, color='red', linestyle='--', alpha=0.5)

axes[1].hist(with_div_cats, bins=range(NUM_CATEGORIES+1), color='steelblue', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Category', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title(f'With Diversity (max 5/cat)\n({len(set(with_div_cats))} unique categories)', fontsize=12)
axes[1].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='Max per category')
axes[1].legend(fontsize=9)

plt.suptitle('Category Distribution in Fused Results', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Analyzing Channel Contributions

In [ ]:
# Analyze which channels contribute to the final results
def analyze_channel_contributions(fused_results, top_k_values=[10, 20, 50, 100]):
    """Analyze channel contribution at different cut-offs."""
    contributions = {}
    
    for k in top_k_values:
        top_k_results = fused_results[:k]
        channel_counts = defaultdict(int)
        
        for item_id, score, channels_str in top_k_results:
            for ch in channels_str.split('+'):
                channel_counts[ch] += 1
        
        contributions[k] = dict(channel_counts)
    
    return contributions

contributions = analyze_channel_contributions(fused, [10, 20, 50, 100, 200])

# Overlap analysis
def compute_channel_overlap(channel_results, top_k=100):
    """Compute pairwise overlap between channels."""
    channel_sets = {}
    for name, results in channel_results.items():
        channel_sets[name] = set([item for item, _ in results[:top_k]])
    
    names = list(channel_sets.keys())
    n = len(names)
    overlap_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            intersection = len(channel_sets[names[i]] & channel_sets[names[j]])
            union = len(channel_sets[names[i]] | channel_sets[names[j]])
            overlap_matrix[i, j] = intersection / union if union > 0 else 0
    
    return overlap_matrix, names

overlap, ch_names = compute_channel_overlap(channel_results, top_k=100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Channel contributions stacked bar
k_values = list(contributions.keys())
ch_list = list(channels.keys())
bottom = np.zeros(len(k_values))

for ch in ch_list:
    values = [contributions[k].get(ch, 0) for k in k_values]
    axes[0].bar(range(len(k_values)), values, bottom=bottom, label=ch, alpha=0.8)
    bottom += np.array(values)

axes[0].set_xticks(range(len(k_values)))
axes[0].set_xticklabels([f'Top-{k}' for k in k_values])
axes[0].set_ylabel('Number of Items', fontsize=11)
axes[0].set_title('Channel Contributions at Different Cut-offs', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# Overlap heatmap
im = axes[1].imshow(overlap, cmap='YlOrRd', vmin=0, vmax=1)
axes[1].set_xticks(range(len(ch_names)))
axes[1].set_yticks(range(len(ch_names)))
axes[1].set_xticklabels(ch_names, rotation=45)
axes[1].set_yticklabels(ch_names)
for i in range(len(ch_names)):
    for j in range(len(ch_names)):
        axes[1].text(j, i, f'{overlap[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=axes[1], label='Jaccard Overlap')
axes[1].set_title('Channel Overlap (Jaccard)', fontsize=12)

plt.tight_layout()
plt.show()

## 7. Meta's Multi-Source Architecture

Meta's production retrieval system uses:

1. **Multiple embedding models**: Different two-tower models optimized for different objectives
2. **Social graph retrieval**: Friends' interactions as a retrieval signal
3. **Cross-entity retrieval**: Retrieve posts, pages, groups — not just items
4. **Dynamic channel weighting**: Learned weights updated based on online metrics
5. **Cascaded fusion**: Coarse fusion → pre-ranking → fine fusion → ranking

> **🔑 Pro Tip:** In production, channel weights are often learned via a lightweight model that takes user context features as input and predicts optimal channel weights. This is essentially a meta-learning problem: learning to combine retrieval channels.

In [ ]:
class LearnedFusionWeights(nn.Module):
    """Learn channel weights from user context."""
    
    def __init__(self, user_feature_dim: int, num_channels: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(user_feature_dim, 32),
            nn.ReLU(),
            nn.Linear(32, num_channels),
            nn.Softmax(dim=-1)
        )
    
    def forward(self, user_features: torch.Tensor) -> torch.Tensor:
        """Predict channel weights for each user."""
        return self.network(user_features)

# Simulate learning channel weights
fusion_model = LearnedFusionWeights(DIM, len(channels))

# Dummy training: optimize weights to maximize "engagement"
optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-3)

# Simulate engagement data
user_features_train = torch.randn(1000, DIM)
# True best channels per user (simulated)
true_best_channels = torch.randint(0, len(channels), (1000,))

losses = []
for epoch in range(50):
    weights_pred = fusion_model(user_features_train)
    loss = F.cross_entropy(weights_pred, true_best_channels)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# Visualize learned weights
fusion_model.eval()
test_users = torch.randn(100, DIM)
with torch.no_grad():
    predicted_weights = fusion_model(test_users).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses, color='teal', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Fusion Weight Learning', fontsize=12)
axes[0].grid(True, alpha=0.3)

ch_names_list = list(channels.keys())
for i, ch_name in enumerate(ch_names_list):
    axes[1].hist(predicted_weights[:, i], bins=20, alpha=0.5, label=ch_name)
axes[1].set_xlabel('Predicted Weight')
axes[1].set_ylabel('Count')
axes[1].set_title('Learned Channel Weight Distribution', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement Dynamic Quota Allocation with Bandit

Use a multi-armed bandit approach to dynamically adjust channel quotas based on observed engagement.

In [ ]:
class BanditQuotaAllocator:
    """Multi-armed bandit for dynamic quota allocation."""
    
    def __init__(self, channel_names, total_budget=500, 
                 exploration_rate=0.1, min_quota=20):
        self.channel_names = channel_names
        self.total_budget = total_budget
        self.exploration_rate = exploration_rate
        self.min_quota = min_quota
        
        # TODO: Initialize UCB counts and rewards per channel
        pass
    
    def allocate(self) -> Dict[str, int]:
        # TODO: Use UCB1 to determine allocation
        #   - Each channel has a "value" based on UCB1
        #   - Allocate proportional to UCB values
        #   - Ensure minimum quota
        pass
    
    def update(self, channel_name: str, reward: float):
        # TODO: Update channel statistics with observed reward
        pass

### 🏋️ Exercise 2: Implement MMR-based Diversified Fusion

Use Maximal Marginal Relevance to select a diverse set of items from the fused pool.

In [ ]:
def mmr_selection(fused_results, item_embeddings, lambda_param=0.7, k=50):
    """
    Maximal Marginal Relevance selection.
    
    MMR(i) = lambda * relevance(i) - (1-lambda) * max_sim(i, selected)
    
    Args:
        fused_results: [(item_id, score, channels), ...]
        item_embeddings: (N, D) item embeddings for diversity
        lambda_param: balance between relevance and diversity
        k: number of items to select
    
    Returns:
        Selected items list
    """
    # TODO: Greedily select items using MMR criterion
    # TODO: First item = highest score
    # TODO: Each subsequent item maximizes MMR
    pass

### 🏋️ Exercise 3: Complete Multi-Channel Pipeline with Evaluation

Build and evaluate the full multi-channel retrieval fusion pipeline.

In [ ]:
def full_pipeline_evaluation(channels, user_ids, ground_truth,
                              total_budget=500, top_k_eval=[10, 50, 100]):
    """
    Complete pipeline evaluation:
    1. Retrieve from all channels
    2. Normalize scores
    3. Fuse with different methods
    4. Apply diversity constraints
    5. Evaluate Recall@K, Category Coverage, Channel Diversity
    
    Compare:
    - Individual channels vs fusion
    - Different normalization methods
    - Different fusion methods
    - Fixed vs dynamic quotas
    
    Returns:
        Comprehensive metrics dict
    """
    # TODO: Run full pipeline for each user
    # TODO: Compare all configuration combinations
    # TODO: Generate visualization of results
    pass

## Summary

| Component | Key Decision | Options |
|-----------|-------------|--------|
| **Channels** | Which retrieval methods to use | CF, Embedding, Content, Trending, Graph, Sequential |
| **Normalization** | How to make scores comparable | Min-Max, Z-Score, Rank-based |
| **Quota** | How many items per channel | Fixed ratios, Dynamic (variance/bandit) |
| **Fusion** | How to combine scores | Max, Average, Weighted, RRF |
| **Diversity** | How to avoid homogeneity | Category caps, Channel caps, MMR |
| **Weights** | How to weight channels | Static, Learned, Contextual |

**Key insight**: Multi-source fusion is where all the retrieval techniques from this part come together. The best systems use 5-10 retrieval channels with dynamic, learned fusion — no single channel dominates.

**This concludes Part 3: Retrieval Systems.** In Part 4, we'll explore the Ranking stage — how to precisely score and order the candidates that retrieval selected.